In [1]:
import cv2,time
from picamera2 import Picamera2
import spidev as SPI
import xgoscreen.LCD_2inch as LCD_2inch
import RPi.GPIO as GPIO
from PIL import Image,ImageDraw,ImageFont
import threading

camera_still=False
class Recode_Video():
    def __init__(self):
        self.display = LCD_2inch.LCD_2inch()
        self.display.Init()
        self.display.clear()
        self.splash = Image.new("RGB",(320,240),"black")
        self.display.ShowImage(self.splash)
        self.draw = ImageDraw.Draw(self.splash)
        self.font = ImageFont.truetype("/home/pi/model/msyh.ttc",15)
        self.picam2=None
        self.camera_still=False
        self.open_camera()

    def open_camera(self):
        if self.picam2==None:
            self.picam2 = Picamera2()
            self.picam2.configure(
                self.picam2.create_preview_configuration(main={"format": "RGB888", "size": (320, 240)})
            )
            self.picam2.start()
    def close_camera(self):
        self.picam2.stop()
        self.picam2.close()

    def xgoCamera(self,switch):
        global camera_still
        if switch:
            #self.open_camera()
            self.camera_still=True
            t = threading.Thread(target=self.camera_mode)  
            t.start() 
        else:
            self.camera_still=False
            time.sleep(0.5)
            splash = Image.new("RGB",(320,240),"black")
            self.display.ShowImage(splash)
            self.close_camera()

    def camera_mode(self):
        self.camera_still=True
        while 1:
            frame = self.picam2.capture_array() 
            image = cv2.flip(frame, 1)
            b,g,r = cv2.split(image)
            image = cv2.merge((r,g,b))
            image = cv2.flip(image,1)
            imgok = Image.fromarray(image)
            self.display.ShowImage(imgok)
            if not self.camera_still:
                break
    def xgoVideoRecord(self,filename="record",seconds=5):
        path="/home/pi/xgoVideos/"
        self.camera_still=False
        time.sleep(0.6)
        FPS=15
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        width = 320
        height = 240
        videoWrite = cv2.VideoWriter(path+filename+'.mp4', fourcc, FPS, (width,height))
        starttime=time.time()
        while 1:
            print('recording...')
            frame = self.picam2.capture_array() 
            image = cv2.flip(frame, 1)
            # if not ret:
            #     break
            videoWrite.write(image)
            b,g,r = cv2.split(image)
            image = cv2.merge((r,g,b))
            image = cv2.flip(image,1)
            imgok = Image.fromarray(image)
            self.display.ShowImage(imgok)
            if time.time()-starttime>seconds:
                break
        print('recording done')
        self.xgoCamera(True)
        videoWrite.release()

In [2]:
import os
import threading
import inspect
import ctypes
video_Path = '/home/pi/xgoVideos/myvidoes.mp4'
sounds_Path = '/home/pi/xgoMusic/myrecord.wav'

#导入xgoedu #Import xguodu
from xgoedu import XGOEDU 
#实例化edu #Instantiate edu
XGO_edu = XGOEDU()#录音对象 Recording object


myXGO_edu = Recode_Video()#录像对象 Video object

Screen already initialized.
Screen already initialized.


[1:07:02.422407273] [36856]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+99-1230f78d
[1:07:02.429742097] [37283]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 29-08-2024 (16:33:32)
[1:07:02.439297780] [37283]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[1:07:02.442684108] [36856]  INFO Camera camera.cpp:1197 configuring streams: (0) 320x240-RGB888 (1) 640x480-GBRG_PISP_COMP1
[1:07:02.442801090] [37283]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected CFE format: 640x480-PC1g


In [3]:
# 关闭线程  stop thread
def _async_raise(tid, exctype):
    """raises the exception, performs cleanup if needed"""
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        # """if it returns a number greater than one, you're in trouble,
        # and you should call it again with exc=NULL to revert the effect"""
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)

In [4]:
#录制

def revideos():
    myXGO_edu.xgoVideoRecord(filename="myvidoes",seconds=5)
def record():     
    XGO_edu.xgoAudioRecord(filename="myrecord",seconds=5) 


thread1 = threading.Thread(target=revideos)
thread1.setDaemon(True)
thread1.start()
record()


/tmp/ipykernel_36856/279176546.py:10: DeprecationWarning: setDaemon() is deprecated, set the daemon attribute instead
  thread1.setDaemon(True)
Recording WAVE '/home/pi/xgoMusic/myrecord.wav' : Signed 32 bit Little Endian, Rate 8000 Hz, Mono


sudo arecord -d 5 -f S32_LE -r 8000 -c 1 -t wav /home/pi/xgoMusic/myrecord.wav
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording..

In [5]:
myXGO_edu.xgoCamera(False)   #关闭摄像头 #Turn off the camera

#合成
cmd = 'ffmpeg -y -i '+video_Path+' -i '+sounds_Path +' -c:v copy -c:a aac -strict experimental -shortest /home/pi/xgoVideos/myoutput.mp4'
#print(cmd)
os.system(cmd)

ffmpeg version 5.1.6-0+deb12u1+rpt3 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1+rpt3 --toolchain=hardened --incdir=/usr/include/aarch64-linux-gnu --enable-gpl --disable-stripping --disable-mmal --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-

0

In [6]:
#会播放到/home/pi/xgoVideos的文件下
#It will play to the file/home/pi/xgoVideos
XGO_edu.xgoVideo(filename="myoutput.mp4")   

/home/pi/xgoVideos/myoutput.mp4
15.0


do_connect: could not connect to socket
connect: No such file or directory
Failed to open LIRC support. You will not be able to use your remote control.
[aac @ 0x7ffec6dd6bc8]Multiple frames in a packet.
AO: [pulse] Init failed: Connection refused
Failed to initialize audio driver 'pulse'
[aac @ 0x7ffec6dd6bc8]channel element 0.0 is not allocated


MPlayer UNKNOWN-12 (C) 2000-2023 MPlayer Team

Playing /home/pi/xgoVideos/myoutput.mp4.
libavformat version 59.27.100 (external)
libavformat file format detected.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7ffec76962d0]Protocol name not provided, cannot determine if input is local or a network protocol, buffers and access patterns cannot be configured optimally without knowing the protocol
[lavf] stream 0: video (mpeg4), -vid 0
[lavf] stream 1: audio (aac), -aid 0, -alang und
Clip info:
 major_brand: isom
 minor_version: 512
 compatible_brands: isomiso2mp41
 encoder: Lavf59.27.100
Load subtitles in /home/pi/xgoVideos/
Opening audio decoder: [ffmpeg] FFmpeg/libavcodec audio decoders
libavcodec version 59.37.100 (external)
AUDIO: 8000 Hz, 1 ch, floatle, 42.1 kbit/16.43% (ratio: 5257->32000)
Selected audio codec: [ffaac] afm: ffmpeg (FFmpeg AAC (MPEG-2/MPEG-4 Audio))
AO: [alsa] 48000Hz 1ch floatle (4 bytes per sample)
Video: no video
Starting playback...
A:   4.9 (04.8) of 5.0 (05.0)  0.1% 


Exiting..

In [7]:
del myXGO_edu
del XGO_edu